# 测试

In [1]:
# import numpy as np
# import tvm
# from tvm.topi import tag
# from tvm import tir as T, te, topi, relay
# @tvm.te.tag_scope(tag=tag.ELEMWISE)
# def round(x):
#     """Round elements of x to nearest integer.

#     Parameters
#     ----------
#     x : tvm.te.Tensor
#         Input argument.

#     Returns
#     -------
#     y : tvm.te.Tensor
#         The result.
#     """
#     def _func(x):
#         return te.round(x)
#     return te.compute(x.shape, _func)

In [2]:
import set_env
import torch
from torchvision.models import resnet18, ResNet18_Weights
from tvm.relay.frontend.pytorch import from_pytorch
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

input_name = "x"
shape = 1, 3, 224, 224
scripted_model = torch.jit.trace(model.eval(), torch.rand(shape).float())
# # 保存模型
# torch.jit.save(scripted_model, './test.pt')
shape_list = [("x", shape)]
relay_mod, params = from_pytorch(scripted_model, shape_list)

In [3]:
from tvm.relay.analysis import extract_intermdeiate_expr

run_mod = extract_intermdeiate_expr(relay_mod, 0)

In [20]:
from tvm.contrib.msc.core.frontend import translate
from tvm.contrib.msc.framework.tvm import codegen as tvm_codegen
from tvm.contrib.msc.framework.torch.frontend import translate as torch_translate
from tvm.contrib.msc.framework.torch import codegen

graph, weights = translate.from_relay(run_mod, params, opt_config={"opt_level": 3})
# model = codegen.to_torch(graph, weights)
# # relay 转 relax
codegen_config = {"explicit_name": False, "from_relay": True}
relax_mod = tvm_codegen.to_relax(graph, weights, codegen_config)

In [24]:
import pandas as pd

In [50]:
df = pd.DataFrame([["D", "3", 12], ["2", 3, 1], [3, 7, 9]])

df.iloc[:, 1]


0    3
1    3
2    7
Name: 1, dtype: object

In [58]:
df.iloc[:, 2].to_numpy()[1:]

array([1, 9])

In [64]:
def f(a=1):
    b = a + 1
    print(b)
    # 函数结束

f(a=2) # 函数调用

3


In [67]:
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights.DEFAULT)
input_name = "data"
shape = 1, 3, 224, 224
trace = torch.jit.trace(model.eval(), torch.rand(shape).float()).eval()
torch.jit.save(trace, "resnet50_v2.pt")